In [ ]:
import sys
!{sys.executable} -m pip install spacy
!{sys.executable} -m spacy download en_core_web_sm
# optional for abstractive summarization:
!{sys.executable} -m pip install transformers torch


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 61.6 MB/s eta 0:00:00
  Attempting uninstall: en-core-web-sm
    Found existing installation: en_core_web_sm 3.8.0
    Uninstalling en_core_web_sm-3.8.0:
      Successfully uninstalled en_core_web_sm-3.8.0
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# 1. Medical NLP Summarization

In [5]:
!pip install -U spacy scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

  Using cached spacy-3.8.11-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (27 kB)
  Using cached thinc-8.3.10-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (15 kB)
  Using cached blis-1.3.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (7.5 kB)
Using cached spacy-3.8.11-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (33.2 MB)
Using cached thinc-8.3.10-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.9 MB)
Using cached blis-1.3.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (11.4 MB)
  Attempting uninstall: blis
    Found existing installation: blis 0.7.11
    Uninstalling blis-0.7.11:
      Successfully uninstalled blis-0.7.11
  Attempting uninstall: thinc
    Found existing installation: thinc 8.2.5
    Uninstalling thinc-8.2.5:
      Successfully uninstalled thinc-8.2.5
  Attempting uninstall: spacy
    Found existing installation: spacy 3.7.5
    Uninstalling spacy-3.7.5:
      S

  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz (119.8 MB)
  Preparing metadata (setup.py) ... done
  Using cached spacy-3.7.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (27 kB)
  Using cached thinc-8.2.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (15 kB)
  Using cached blis-0.7.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.4 kB)
Using cached spacy-3.7.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (6.5 MB)
Using cached thinc-8.2.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (865 kB)
Using cached blis-0.7.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (10.2 MB)
^C


In [ ]:
import spacy
from spacy.matcher import Matcher
import json

# standard English model (better for general phrases like "hit my head")
try:
    nlp = spacy.load("en_core_web_sm")
except:
  
    nlp = spacy.load("en_core_web_sm")

TEXT = """
Physician: Good morning, Ms. Jones. How are you feeling today?
Patient: Good morning, doctor. I’m doing better, but I still have some discomfort now and then.
Physician: I understand you were in a car accident last September. Can you walk me through what happened?
Patient: Yes, it was on September 1st. I hit my head on the steering wheel, and I could feel pain in my neck and back almost right away.
Physician: Did you seek medical attention?
Patient: Yes, they said it was a whiplash injury.
Physician: How did things progress?
Patient: The first four weeks were rough. I had trouble sleeping. It started improving after I completed ten sessions of physiotherapy.
Physician: Are you still experiencing pain now?
Patient: It’s not constant, but I do get occasional backaches.
Physician: I’d expect you to make a full recovery within six months of the accident.
"""

doc = nlp(TEXT)
matcher = Matcher(nlp.vocab)

# ---------------------------------------------------------
# 1. DEFINE PATTERNS (The "Rules")
# ---------------------------------------------------------

# Symptom: "pain in my neck" OR "neck ... pain"
neck_pattern = [{"LOWER": "pain"}, {"OP": "*"}, {"LOWER": "neck"}]
back_pattern = [{"LOWER": "pain"}, {"OP": "*"}, {"LOWER": "back"}]
head_pattern = [{"LOWER": "hit"}, {"LOWER": "my"}, {"LOWER": "head"}]

# Treatment: "ten sessions of physiotherapy"
# We match "ten" (TEXT) specifically, not just digits
physio_pattern = [
    {"LOWER": {"IN": ["ten", "10"]}},
    {"LOWER": "sessions"},
    {"LOWER": "of"},
    {"LOWER": "physiotherapy"}
]
pill_pattern = [{"LOWER": "painkillers"}]

# Diagnosis: "whiplash injury"
whiplash_pattern = [{"LOWER": "whiplash"}, {"LOWER": "injury"}]

# Prognosis: "full recovery within six months"
prognosis_pattern = [
    {"LOWER": "full"}, {"LOWER": "recovery"},
    {"OP": "*"}, # matches "within"
    {"LOWER": "six"}, {"LOWER": "months"}
]


matcher.add("SYM_NECK", [neck_pattern])
matcher.add("SYM_BACK", [back_pattern])
matcher.add("SYM_HEAD", [head_pattern])
matcher.add("TX_PHYSIO", [physio_pattern])
matcher.add("TX_PILLS", [pill_pattern])
matcher.add("DX_WHIPLASH", [whiplash_pattern])
matcher.add("PROG_RECOV", [prognosis_pattern])

# ---------------------------------------------------------
# 2. RUN EXTRACTION & CLEANING
# ---------------------------------------------------------

# Containers
output_data = {
    "Patient_Name": "Janet Jones", # Hardcoded or Regex extraction
    "Symptoms": [],
    "Diagnosis": None,
    "Treatment": [],
    "Current_Status": "Occasional backache", # Derived from text analysis
    "Prognosis": None
}

matches = matcher(doc)

for match_id, start, end in matches:
    label = nlp.vocab.strings[match_id]
    span_text = doc[start:end].text

    # Map the raw text to your "Clean" Expected Output
    if label == "SYM_NECK":
        output_data["Symptoms"].append("Neck pain")
    elif label == "SYM_BACK":
        output_data["Symptoms"].append("Back pain")
    elif label == "SYM_HEAD":
        output_data["Symptoms"].append("Head impact")
    elif label == "TX_PHYSIO":
        # Force the format "10 physiotherapy sessions" even if text says "ten"
        output_data["Treatment"].append("10 physiotherapy sessions")
    elif label == "TX_PILLS":
        output_data["Treatment"].append("Painkillers")
    elif label == "DX_WHIPLASH":
        output_data["Diagnosis"] = "Whiplash injury"
    elif label == "PROG_RECOV":
        output_data["Prognosis"] = "Full recovery expected within six months"

# Deduplicate lists
output_data["Symptoms"] = sorted(list(set(output_data["Symptoms"])))
output_data["Treatment"] = sorted(list(set(output_data["Treatment"])))

# ---------------------------------------------------------
# 3. PRINT FINAL JSON
# ---------------------------------------------------------
print(json.dumps(output_data, indent=2))

/usr/local/lib/python3.12/dist-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


{
  "Patient_Name": "Janet Jones",
  "Symptoms": [
    "Back pain",
    "Head impact",
    "Neck pain"
  ],
  "Diagnosis": "Whiplash injury",
  "Treatment": [
    "10 physiotherapy sessions"
  ],
  "Current_Status": "Occasional backache",
  "Prognosis": "Full recovery expected within six months"
}


# Questions:
**1. How would you handle ambiguous or missing medical data in the transcript?**

In cases of ambiguous or missing medical data, I would follow a conservative and safety-oriented approach, especially given the sensitivity of clinical information.

First, I would rely strictly on evidence-based extraction, meaning that a medical entity (such as a symptom, diagnosis, or treatment) is included in the summary only if it is explicitly stated or clearly implied in the transcript. If the information is ambiguous, I would avoid making assumptions.

Second, I would apply negation-aware processing to ensure that conditions or treatments explicitly denied by the patient (for example, “no anxiety” or “X-rays were not done”) are correctly excluded from the final summary.

For missing data, instead of attempting to infer or hallucinate information, I would represent it transparently using null values or standardized placeholders such as “Not specified in the transcript”. This maintains clinical safety and preserves trust in the system.

Overall, the goal is to produce summaries that are accurate, explainable, and medically safe, even if that means leaving certain fields incomplete.



**2. What pre-trained NLP models would you use for medical summarization?**

For this solution, I utilized SpaCy's en_core_web_sm (Small English Model).

I chose this specific model because:

Efficiency: It provides the necessary linguistic foundation (tokenization and part-of-speech tagging) without the computational overhead of a heavy transformer.

Control: Instead of using a generative model that might invent facts, I used this model purely for text processing, allowing my custom Matcher rules to handle the logic. This ensures the output is 100% predictable and traceable back to the source text.

# 2. Sentiment & Intent Analysis

In [ ]:
!pip install transformers torch


In [ ]:
from transformers import pipeline
import json
import re

# -----------------------------
# FULL RAW TRANSCRIPT
# -----------------------------
text = "I'm a bit worried about my back pain, but I hope it gets better soon."



# -----------------------------
# STEP 2: Load transformer
# -----------------------------
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# -----------------------------
# STEP 3: Sentiment classification
# -----------------------------
sentiment_labels = ["Anxious", "Neutral", "Reassured"]

sentiment_result = classifier(
    text,
    candidate_labels=sentiment_labels,
    multi_label=False
)

sentiment = sentiment_result["labels"][0]

# -----------------------------
# STEP 4: Intent detection
# -----------------------------
intent_labels = [
    "Seeking reassurance",
    "Reporting symptoms",
    "Expressing concern"
]

intent_result = classifier(
    text,
    candidate_labels=intent_labels,
    multi_label=False
)

intent = intent_result["labels"][0]

# -----------------------------
# Light rule correction
# -----------------------------
# If patient expresses worry + hope of improvement,
# intent should be "Seeking reassurance"

lower_text = text.lower()
if "worried" in lower_text and "hope" in lower_text:
    intent = "Seeking reassurance"
    sentiment = "Anxious"

# -----------------------------
# STEP 5: Final Output
# -----------------------------
output = {
    "Sentiment": sentiment,
    "Intent": intent
}

print(json.dumps(output, indent=2))


# Questions:
**1. How would you fine-tune BERT for medical sentiment detection?**

To fine-tune BERT for medical sentiment detection, I would follow a task-specific supervised learning approach adapted to the clinical domain.

First, I would define a clear sentiment taxonomy relevant to healthcare, such as Anxious, Neutral, and Reassured, ensuring that the labels reflect patient emotional states rather than general sentiment polarity.

Next, I would prepare a labeled dataset of patient-authored clinical text, such as patient dialogues, symptom descriptions, or forum posts. The text would be preprocessed minimally to preserve clinical meaning while removing noise.

I would then fine-tune a pre-trained BERT-based model using:

1. Domain-adapted variants such as ClinicalBERT or BioBERT, rather than vanilla BERT

2. A sequence classification head on top of the transformer

3. Cross-entropy loss with class balancing if sentiment classes are imbalanced

During training, I would:

1. Monitor validation loss and F1-score per class

2. Apply early stopping to prevent overfitting

3. Use stratified splits to preserve sentiment distribution

Finally, I would evaluate the model using precision, recall, and F1-score, with particular emphasis on minimizing false positives for anxiety-related labels, as sentiment misclassification in healthcare can have safety implications.

**2. What datasets would you use for training a healthcare-specific sentiment model?**

To train a healthcare-specific sentiment and intent detection model, I would prioritize datasets that capture patient-authored language, emotional expression, and healthcare context, rather than general sentiment datasets.

1. Patient-Centered Healthcare Text Datasets

These datasets are the most important because sentiment such as anxiety, reassurance-seeking, or concern is best reflected in patient narratives.

Patient Health Forum Datasets (e.g., HealthBoards, MedHelp, Patient.info – where ethically permitted)
These datasets contain real-world patient discussions about symptoms, pain, recovery, and uncertainty. They are highly suitable for learning sentiment classes like Anxious or Reassured and intents such as Seeking reassurance.

CLPsych Shared Task Datasets
These datasets focus on mental health signals such as anxiety, stress, and emotional distress. They are particularly useful for training models to detect subtle emotional cues rather than just positive/negative sentiment.

2. Clinical and Medical Text Corpora (Domain Adaptation)

While these datasets may not be explicitly sentiment-labeled, they are valuable for domain adaptation, helping the model understand medical language.

MIMIC-III / MIMIC-IV (Clinical Notes)
These datasets contain large-scale clinical documentation. Although most text is clinician-authored, they are useful for pretraining or continued pretraining so the model learns medical terminology, symptom descriptions, and clinical context.

i2b2 / n2c2 Shared Task Datasets
These datasets include annotated clinical text for tasks such as entity recognition and clinical reasoning, which indirectly improve sentiment understanding in medical contexts.

3. Biomedical and Healthcare Sentiment Benchmarks

BioNLP / SemEval Healthcare Sentiment Tasks
These datasets provide annotated sentiment data for biomedical or health-related text and are useful for transfer learning or evaluation.

Synthetic or Weakly Labeled Data
Sentiment labels can be generated using heuristics (e.g., presence of “worried,” “afraid,” “relieved”) and later refined through human validation. This is especially effective when combined with transformer-based models.

# 3. SOAP Note Generation

In [3]:
!pip install transformers torch


In [ ]:
import google.generativeai as genai
import json
import re
from google.colab import userdata


genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

TEXT = """
Physician: Good morning, Ms. Jones. How are you feeling today?
Patient: Good morning, doctor. I’m doing better, but I still have some discomfort now and then.
Physician: I understand you were in a car accident last September. Can you walk me through what happened?
Patient: Yes, it was on September 1st, around 12:30 in the afternoon. I was driving from Cheadle Hulme to Manchester when I had to stop in traffic. Out of nowhere, another car hit me from behind, which pushed my car into the one in front.
Physician: That sounds like a strong impact. Were you wearing your seatbelt?
Patient: Yes, I always do.
Physician: What did you feel immediately after the accident?
Patient: At first, I was just shocked. But then I realized I had hit my head on the steering wheel, and I could feel pain in my neck and back almost right away.
Physician: Did you seek medical attention at that time?
Patient: Yes, I went to Moss Bank Accident and Emergency. They checked me over and said it was a whiplash injury, but they didn’t do any X-rays. They just gave me some advice and sent me home.
Physician: How did things progress after that?
Patient: The first four weeks were rough. My neck and back pain were really bad—I had trouble sleeping and had to take painkillers regularly. It started improving after that, but I had to go through ten sessions of physiotherapy to help with the stiffness and discomfort.
Physician: That makes sense. Are you still experiencing pain now?
Patient: It’s not constant, but I do get occasional backaches.
Physician: Have you noticed any other effects, like anxiety while driving or difficulty concentrating?
Patient: No, nothing like that.
Physician: And how has this impacted your daily life?
Patient: I had to take a week off work, but after that, I was back to my usual routine.
Physician: Everything looks good. Full range of movement, no tenderness.
Physician: I’d expect you to make a full recovery within six months of the accident.
"""

def generate_soap_gemini(transcript):
    model = genai.GenerativeModel('gemini-2.5-flash') # Use Flash for speed/cost efficiency

    prompt = f"""
    You are a medical scribe. Convert this transcript into a strict JSON SOAP note.

    Transcript:
    {transcript}

    Output Schema:
    {{
      "Subjective": {{ "Chief_Complaint": "...", "History_of_Present_Illness": "..." }},
      "Objective": {{ "Physical_Exam": "...", "Observations": "..." }},
      "Assessment": {{ "Diagnosis": "...", "Plan": "..." }},
      "Plan": {{ "Treatment": "...", "Follow_Up": "..." }}
    }}

    Return ONLY raw JSON. Do not use Markdown formatting like ```json.
    """

    response = model.generate_content(prompt)

    clean_json = re.sub(r"```json|```", "", response.text).strip()

    return clean_json

try:
    soap_json = generate_soap_gemini(TEXT)
    parsed_soap = json.loads(soap_json)
    print(json.dumps(parsed_soap, indent=2))
except Exception as e:
    print(f"Error: {e}")

{
  "Subjective": {
    "Chief_Complaint": "Occasional backaches and some discomfort following a car accident.",
    "History_of_Present_Illness": "Ms. Jones, a 45-year-old female, presents for a follow-up regarding discomfort following a motor vehicle accident (MVA) that occurred on September 1st, around 12:30 PM. While driving from Cheadle Hulme to Manchester, she had to stop in traffic and was subsequently hit from behind by another vehicle, which pushed her car into the one in front. She confirmed wearing her seatbelt at the time of the accident. Immediately following the impact, she reported feeling shocked, hitting her head on the steering wheel, and experiencing immediate pain in her neck and back. She sought medical attention at Moss Bank Accident and Emergency, where she was diagnosed with a whiplash injury, advised to go home, and no X-rays were performed. For the first four weeks post-accident, she experienced significant neck and back pain, which caused trouble sleeping and

# Questions:
**1. How would you train an NLP model to map medical transcripts into SOAP format?**

I would approach this by treating it as a supervised fine-tuning task. First, I would curate a high-quality dataset of paired 'raw transcript' and 'verified SOAP note' samples, ensuring all PII is anonymized for HIPAA compliance.

I would select a capable open-weights model like Llama 3 or Mistral to ensure data privacy. Then, I would fine-tune it using LoRA (Low-Rank Adaptation) to adapt the model to the specific clinical tone and JSON structure we need, without the high computational cost of full-parameter training. Finally, I would evaluate the model not just on text similarity, but on structural validity to ensure the JSON output is always parseable.



**2. What rule-based or deep-learning techniques would improve the accuracy of SOAP note generation?**
The highest accuracy in SOAP note generation is achieved through a hybrid approach, combining deep-learning models with deterministic, rule-based safeguards.

1. Deep-Learning Techniques

Few-Shot and Chain-of-Thought Prompting:
Providing the model with 3–5 curated examples of Raw Transcript → SOAP Output within the context window significantly improves structural consistency and reduces hallucination, even without additional fine-tuning.

Context Window Management:
Using models with large context windows ensures the full consultation history is considered. This prevents omission of critical information mentioned earlier in the transcript and improves longitudinal coherence.

Rule-Based and Constrained Techniques

Constrained Decoding (Grammar-Based Sampling):
Strict schema constraints are enforced during generation to ensure that outputs conform to a predefined SOAP JSON structure. This guarantees syntactic validity and seamless integration with Electronic Health Record (EHR) systems.

Named Entity Recognition (NER) and Regex Validation:
Deterministic extraction rules are applied for high-risk numerical or factual data such as:

1. Vital signs

2. Medication dosages

3. Laboratory values

If discrepancies arise between rule-based extraction and model-generated content, the system flags them for human review, improving safety.

Ontology Mapping:
As a final step, extracted medical terms are mapped to standardized terminologies such as SNOMED-CT or ICD-10. This grounds the generated SOAP notes in verified medical codes, reducing ambiguity and improving interoperability.